# 🔍 Logo Detection with GAN Augmentation + YOLOv8
### Full Training Pipeline — Kaggle GPU Notebook

**Pipeline:**
```
Raw Data → Preprocessing → DCGAN Training → Synthetic Images → Merge Dataset → YOLOv8 Training → Evaluation
```

**Dataset:** FlickrLogos-32  
**GAN:** DCGAN (PyTorch)  
**Detector:** YOLOv8n (Ultralytics)


## 📦 Step 1: Install Dependencies

In [ ]:
!pip install ultralytics==8.2.0 -q
!pip install opencv-python-headless matplotlib seaborn Pillow tqdm -q
print('✅ All packages installed')

## 📥 Step 2: Download & Prepare Dataset

In [ ]:
import os
import json
import shutil
import random
import zipfile
import urllib.request
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.utils as vutils

# ─── Paths ────────────────────────────────────────────────────────────────────
BASE        = Path('/kaggle/working')
DATA_DIR    = BASE / 'dataset'
RAW_DIR     = DATA_DIR / 'raw'
GAN_OUT     = BASE / 'generated'
YOLO_DIR    = DATA_DIR / 'yolo'
ORIG_RUN    = BASE / 'runs' / 'original'
AUG_RUN     = BASE / 'runs' / 'augmented'

for p in [RAW_DIR, GAN_OUT, YOLO_DIR/'images'/'train',
          YOLO_DIR/'images'/'val', YOLO_DIR/'labels'/'train',
          YOLO_DIR/'labels'/'val', ORIG_RUN, AUG_RUN]:
    p.mkdir(parents=True, exist_ok=True)

print('✅ Directories created')

In [ ]:
# ─── FlickrLogos-32 Download ──────────────────────────────────────────────────
# Option A: Use Kaggle dataset (recommended)
# Run this in a Kaggle terminal or add the dataset via 'Add Data' panel:
#   kaggle datasets download -d shahraizanwar/flickrlogos32 -p /kaggle/working/dataset/raw --unzip
#
# Option B: Direct download (if available)
# The cell below creates a synthetic mini-dataset for demonstration when
# the real dataset is not mounted. Replace this with your actual data path.

FLICKR_DATASET_PATH = Path('/kaggle/input/flickrlogos32')  # adjust if needed

USE_SYNTHETIC_DEMO = not FLICKR_DATASET_PATH.exists()
if USE_SYNTHETIC_DEMO:
    print('⚠️  FlickrLogos-32 not found — generating synthetic demo data.')
    print('    Add the dataset via Kaggle "Add Data" > flickrlogos32')
else:
    print(f'✅ Dataset found at {FLICKR_DATASET_PATH}')

In [ ]:
# ─── Logo Classes (FlickrLogos-32 subset — top 10 for speed) ──────────────────
LOGO_CLASSES = [
    'adidas', 'apple', 'bmw', 'cocacola', 'fedex',
    'ferrari', 'ford', 'google', 'gucci', 'hp'
]
NUM_CLASSES = len(LOGO_CLASSES)
CLASS2IDX   = {c: i for i, c in enumerate(LOGO_CLASSES)}
print(f'Classes ({NUM_CLASSES}):', LOGO_CLASSES)

In [ ]:
def generate_synthetic_demo_dataset(
    out_images, out_labels, n_per_class=80, img_size=640
):
    """Create synthetic coloured-rectangle 'logos' for pipeline testing."""
    np.random.seed(42)
    color_map = {
        'adidas': (0,0,0), 'apple': (180,180,180), 'bmw': (0,0,200),
        'cocacola': (200,0,0), 'fedex': (100,0,200), 'ferrari': (220,0,0),
        'ford': (0,0,160), 'google': (0,160,0), 'gucci': (20,80,20),
        'hp': (0,100,200)
    }
    created = 0
    for cls_name, cls_idx in CLASS2IDX.items():
        color = color_map[cls_name]
        for i in range(n_per_class):
            img = np.ones((img_size, img_size, 3), dtype=np.uint8) * 240
            # random background texture
            noise = np.random.randint(0, 30, img.shape, dtype=np.uint8)
            img = np.clip(img.astype(int) - noise, 0, 255).astype(np.uint8)

            # logo bounding box
            w = np.random.randint(80, 200)
            h = np.random.randint(40, 120)
            x1 = np.random.randint(0, img_size - w)
            y1 = np.random.randint(0, img_size - h)
            cv2.rectangle(img, (x1, y1), (x1+w, y1+h), color, -1)
            # text
            cv2.putText(img, cls_name[:3].upper(), (x1+5, y1+h-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)

            fname = f'{cls_name}_{i:04d}.jpg'
            cv2.imwrite(str(out_images / fname), img)

            # YOLO label: <cls> <cx> <cy> <w> <h>  (normalized)
            cx = (x1 + w/2) / img_size
            cy = (y1 + h/2) / img_size
            nw = w / img_size
            nh = h / img_size
            lbl_path = out_labels / (fname.replace('.jpg', '.txt'))
            with open(lbl_path, 'w') as f:
                f.write(f'{cls_idx} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n')
            created += 1
    print(f'✅ Synthetic dataset: {created} images created')


def convert_flickr_to_yolo(flickr_root, out_images, out_labels):
    """Convert FlickrLogos-32 annotations to YOLO format."""
    flickr_root = Path(flickr_root)
    converted = 0
    for cls_name in LOGO_CLASSES:
        cls_dir = flickr_root / cls_name
        if not cls_dir.exists():
            continue
        cls_idx = CLASS2IDX[cls_name]
        ann_files = list(cls_dir.glob('*.gt_data.txt'))
        for ann_file in ann_files:
            img_name = ann_file.stem.replace('.gt_data', '') + '.jpg'
            img_src  = cls_dir / img_name
            if not img_src.exists():
                continue
            img = cv2.imread(str(img_src))
            if img is None:
                continue
            ih, iw = img.shape[:2]
            dst_img = out_images / img_name
            shutil.copy(img_src, dst_img)

            lbl_path = out_labels / img_name.replace('.jpg', '.txt')
            with open(ann_file) as f, open(lbl_path, 'w') as lf:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    x1, y1, x2, y2 = map(int, parts[1:5])
                    cx = ((x1+x2)/2) / iw
                    cy = ((y1+y2)/2) / ih
                    nw = (x2-x1)    / iw
                    nh = (y2-y1)    / ih
                    lf.write(f'{cls_idx} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n')
            converted += 1
    print(f'✅ Converted {converted} FlickrLogos-32 images to YOLO format')


# ─── Run appropriate loader ───────────────────────────────────────────────────
ALL_IMAGES  = DATA_DIR / 'all_images'
ALL_LABELS  = DATA_DIR / 'all_labels'
ALL_IMAGES.mkdir(exist_ok=True)
ALL_LABELS.mkdir(exist_ok=True)

if USE_SYNTHETIC_DEMO:
    generate_synthetic_demo_dataset(ALL_IMAGES, ALL_LABELS, n_per_class=80)
else:
    convert_flickr_to_yolo(FLICKR_DATASET_PATH, ALL_IMAGES, ALL_LABELS)

all_imgs = sorted(ALL_IMAGES.glob('*.jpg'))
print(f'Total images: {len(all_imgs)}')

In [ ]:
# ─── Train / Val Split ────────────────────────────────────────────────────────
random.seed(42)
random.shuffle(all_imgs)
split = int(0.8 * len(all_imgs))
train_imgs, val_imgs = all_imgs[:split], all_imgs[split:]

def copy_split(img_list, img_dst, lbl_dst):
    for src in img_list:
        shutil.copy(src, img_dst / src.name)
        lbl_src = ALL_LABELS / src.with_suffix('.txt').name
        if lbl_src.exists():
            shutil.copy(lbl_src, lbl_dst / lbl_src.name)

copy_split(train_imgs, YOLO_DIR/'images'/'train', YOLO_DIR/'labels'/'train')
copy_split(val_imgs,   YOLO_DIR/'images'/'val',   YOLO_DIR/'labels'/'val')
print(f'Train: {len(train_imgs)} | Val: {len(val_imgs)}')

In [ ]:
# ─── Write YOLO dataset.yaml ──────────────────────────────────────────────────
yaml_content = f"""path: {YOLO_DIR}
train: images/train
val:   images/val

nc: {NUM_CLASSES}
names: {LOGO_CLASSES}
"""
yaml_path = YOLO_DIR / 'dataset.yaml'
yaml_path.write_text(yaml_content)
print(yaml_path.read_text())

## 🧠 Step 3: DCGAN Training

In [ ]:
# ─── DCGAN Architecture ───────────────────────────────────────────────────────
IMG_SIZE   = 64   # GAN output resolution
NZ         = 100  # latent vector size
NGF        = 64   # generator feature maps
NDF        = 64   # discriminator feature maps
NC         = 3    # RGB channels
BATCH_SIZE = 64
GAN_EPOCHS = 25   # keep low for Kaggle time limits
LR         = 0.0002
BETA1      = 0.5
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


def weights_init(m):
    """Custom weight initialisation for DCGAN (from original paper)."""
    cname = m.__class__.__name__
    if 'Conv' in cname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in cname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # input: NZ × 1 × 1
            nn.ConvTranspose2d(NZ, NGF*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(NGF*8), nn.ReLU(True),
            # → NGF*8 × 4 × 4
            nn.ConvTranspose2d(NGF*8, NGF*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF*4), nn.ReLU(True),
            # → NGF*4 × 8 × 8
            nn.ConvTranspose2d(NGF*4, NGF*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF*2), nn.ReLU(True),
            # → NGF*2 × 16 × 16
            nn.ConvTranspose2d(NGF*2, NGF,   4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF),   nn.ReLU(True),
            # → NGF × 32 × 32
            nn.ConvTranspose2d(NGF, NC, 4, 2, 1, bias=False),
            nn.Tanh()
            # → NC × 64 × 64
        )
    def forward(self, x):
        return self.net(x)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # input: NC × 64 × 64
            nn.Conv2d(NC, NDF, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(NDF, NDF*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF*2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(NDF*2, NDF*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF*4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(NDF*4, NDF*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF*8), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(NDF*8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).view(-1)


netG = Generator().to(DEVICE).apply(weights_init)
netD = Discriminator().to(DEVICE).apply(weights_init)
print('Generator params:', sum(p.numel() for p in netG.parameters()))
print('Discriminator params:', sum(p.numel() for p in netD.parameters()))

In [ ]:
# ─── Logo Dataset for GAN ─────────────────────────────────────────────────────
class LogoGANDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.paths = list(Path(img_dir).glob('*.jpg'))
        self.transform = transform or transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.2, 0.2, 0.2),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img)


gan_dataset = LogoGANDataset(ALL_IMAGES)
gan_loader  = DataLoader(gan_dataset, batch_size=BATCH_SIZE,
                         shuffle=True, num_workers=2, pin_memory=True)
print(f'GAN dataset size: {len(gan_dataset)}')

In [ ]:
# ─── DCGAN Training Loop ──────────────────────────────────────────────────────
criterion = nn.BCELoss()
optimD = optim.Adam(netD.parameters(), lr=LR, betas=(BETA1, 0.999))
optimG = optim.Adam(netG.parameters(), lr=LR, betas=(BETA1, 0.999))

fixed_noise = torch.randn(64, NZ, 1, 1, device=DEVICE)
real_label, fake_label = 1.0, 0.0

G_losses, D_losses = [], []

print('Starting DCGAN training...')
for epoch in range(GAN_EPOCHS):
    g_loss_ep = d_loss_ep = 0.0
    for i, real_imgs in enumerate(gan_loader):
        real_imgs = real_imgs.to(DEVICE)
        bs = real_imgs.size(0)

        # ── Train Discriminator ──
        netD.zero_grad()
        label = torch.full((bs,), real_label, device=DEVICE)
        out   = netD(real_imgs)
        errD_real = criterion(out, label)
        errD_real.backward()

        noise    = torch.randn(bs, NZ, 1, 1, device=DEVICE)
        fake     = netG(noise)
        label.fill_(fake_label)
        out      = netD(fake.detach())
        errD_fake = criterion(out, label)
        errD_fake.backward()
        errD = errD_real + errD_fake
        optimD.step()

        # ── Train Generator ──
        netG.zero_grad()
        label.fill_(real_label)
        out  = netD(fake)
        errG = criterion(out, label)
        errG.backward()
        optimG.step()

        g_loss_ep += errG.item()
        d_loss_ep += errD.item()

    avg_g = g_loss_ep / len(gan_loader)
    avg_d = d_loss_ep / len(gan_loader)
    G_losses.append(avg_g)
    D_losses.append(avg_d)
    print(f'Epoch [{epoch+1:02d}/{GAN_EPOCHS}]  Loss_D: {avg_d:.4f}  Loss_G: {avg_g:.4f}')

print('✅ GAN training complete')

In [ ]:
# ─── Plot GAN Losses ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(G_losses, label='Generator',     color='#e63946', linewidth=2)
axes[0].plot(D_losses, label='Discriminator', color='#457b9d', linewidth=2)
axes[0].set_title('GAN Training Losses', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Generated sample grid
with torch.no_grad():
    gen_sample = netG(fixed_noise).detach().cpu()
grid = vutils.make_grid(gen_sample, nrow=8, normalize=True)
axes[1].imshow(grid.permute(1,2,0).numpy())
axes[1].set_title('GAN Generated Logos', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.savefig(BASE / 'gan_training_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Loss plot saved')

In [ ]:
# ─── Save GAN weights ─────────────────────────────────────────────────────────
torch.save(netG.state_dict(), BASE / 'generator.pth')
torch.save(netD.state_dict(), BASE / 'discriminator.pth')
print('✅ GAN weights saved')

## 🖼️ Step 4: Generate Synthetic Images for Augmentation

In [ ]:
# ─── Generate & save synthetic images ────────────────────────────────────────
GENERATE_COUNT = 500   # total synthetic images to produce
YOLO_GEN_SIZE  = 640   # upscale to match YOLO training size

netG.eval()
gen_image_dir = GAN_OUT / 'images'
gen_label_dir = GAN_OUT / 'labels'
gen_image_dir.mkdir(exist_ok=True)
gen_label_dir.mkdir(exist_ok=True)

saved = 0
with torch.no_grad():
    while saved < GENERATE_COUNT:
        batch = min(64, GENERATE_COUNT - saved)
        noise = torch.randn(batch, NZ, 1, 1, device=DEVICE)
        fakes = netG(noise).cpu()

        for j in range(batch):
            # denormalize [-1,1] → [0,255]
            img_t = fakes[j] * 0.5 + 0.5
            img_np = (img_t.permute(1,2,0).numpy() * 255).astype(np.uint8)
            img_np = cv2.resize(img_np, (YOLO_GEN_SIZE, YOLO_GEN_SIZE))

            fname = f'gen_{saved:05d}.jpg'
            cv2.imwrite(str(gen_image_dir / fname),
                        cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR))

            # Assign pseudo-label: centred bounding box (full image is the logo)
            cls_idx = saved % NUM_CLASSES
            lbl_path = gen_label_dir / fname.replace('.jpg', '.txt')
            with open(lbl_path, 'w') as f:
                f.write(f'{cls_idx} 0.500000 0.500000 0.800000 0.800000\n')

            saved += 1

print(f'✅ {saved} synthetic images saved to {gen_image_dir}')

In [ ]:
# ─── Visualise 16 generated samples ──────────────────────────────────────────
gen_samples = sorted(gen_image_dir.glob('*.jpg'))[:16]
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, p in zip(axes.flat, gen_samples):
    img = cv2.imread(str(p))
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.axis('off')
    ax.set_title(p.stem, fontsize=7)
plt.suptitle('GAN-Generated Logo Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE / 'generated_samples.png', dpi=120)
plt.show()

## 🎯 Step 5: Train YOLOv8 — Original Dataset

In [ ]:
from ultralytics import YOLO

# ─── Train on original data ───────────────────────────────────────────────────
model_orig = YOLO('yolov8n.pt')   # nano — fastest

results_orig = model_orig.train(
    data    = str(yaml_path),
    epochs  = 30,
    imgsz   = 640,
    batch   = 16,
    device  = 0 if torch.cuda.is_available() else 'cpu',
    project = str(BASE / 'runs'),
    name    = 'original',
    exist_ok= True,
    patience= 10,
    verbose = True
)

orig_best = BASE / 'runs' / 'original' / 'weights' / 'best.pt'
print(f'✅ Original model saved: {orig_best}')

## 🎯 Step 6: Train YOLOv8 — GAN-Augmented Dataset

In [ ]:
# ─── Merge original + generated images into augmented split ──────────────────
AUG_YOLO_DIR = DATA_DIR / 'yolo_augmented'
for sub in ['images/train','images/val','labels/train','labels/val']:
    (AUG_YOLO_DIR / sub).mkdir(parents=True, exist_ok=True)

# Copy original splits
for sub in ['images/train','images/val','labels/train','labels/val']:
    for f in (YOLO_DIR / sub).iterdir():
        shutil.copy(f, AUG_YOLO_DIR / sub / f.name)

# Add generated images to train only
for img_f in gen_image_dir.glob('*.jpg'):
    lbl_f = gen_label_dir / img_f.with_suffix('.txt').name
    shutil.copy(img_f, AUG_YOLO_DIR / 'images' / 'train' / img_f.name)
    if lbl_f.exists():
        shutil.copy(lbl_f, AUG_YOLO_DIR / 'labels' / 'train' / lbl_f.name)

aug_yaml = AUG_YOLO_DIR / 'dataset.yaml'
aug_yaml.write_text(f"""path: {AUG_YOLO_DIR}
train: images/train
val:   images/val
nc: {NUM_CLASSES}
names: {LOGO_CLASSES}
""")

n_aug_train = len(list((AUG_YOLO_DIR/'images'/'train').iterdir()))
print(f'Augmented train set size: {n_aug_train}')

In [ ]:
# ─── Train on GAN-augmented data ──────────────────────────────────────────────
model_aug = YOLO('yolov8n.pt')

results_aug = model_aug.train(
    data    = str(aug_yaml),
    epochs  = 30,
    imgsz   = 640,
    batch   = 16,
    device  = 0 if torch.cuda.is_available() else 'cpu',
    project = str(BASE / 'runs'),
    name    = 'augmented',
    exist_ok= True,
    patience= 10,
    verbose = True
)

aug_best = BASE / 'runs' / 'augmented' / 'weights' / 'best.pt'
# Copy best model to known path
shutil.copy(aug_best, BASE / 'best.pt')
print(f'✅ Augmented model saved: {BASE / "best.pt"}')

## 📊 Step 7: Compare Results

In [ ]:
# ─── Extract metrics and compare ──────────────────────────────────────────────
import csv

def load_results_csv(run_dir):
    """Load ultralytics results.csv and return last-epoch metrics."""
    csv_path = Path(run_dir) / 'results.csv'
    if not csv_path.exists():
        return {}
    with open(csv_path) as f:
        rows = list(csv.DictReader(f))
    if not rows:
        return {}
    last = rows[-1]
    return {k.strip(): float(v) for k, v in last.items() if v.strip()}

orig_metrics = load_results_csv(BASE / 'runs' / 'original')
aug_metrics  = load_results_csv(BASE / 'runs' / 'augmented')

metric_keys = [
    'metrics/mAP50(B)', 'metrics/mAP50-95(B)',
    'metrics/precision(B)', 'metrics/recall(B)'
]
labels_pretty = ['mAP@50', 'mAP@50-95', 'Precision', 'Recall']

orig_vals = [orig_metrics.get(k, 0) for k in metric_keys]
aug_vals  = [aug_metrics.get(k, 0) for k in metric_keys]

x = np.arange(len(labels_pretty))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - w/2, orig_vals, w, label='Original',     color='#457b9d', alpha=0.9)
bars2 = ax.bar(x + w/2, aug_vals,  w, label='GAN-Augmented',color='#e63946', alpha=0.9)

ax.set_title('YOLOv8 Performance: Original vs GAN-Augmented',
             fontsize=14, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(labels_pretty)
ax.set_ylim(0, 1); ax.set_ylabel('Score')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)

for bar in bars1 + bars2:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(BASE / 'comparison_chart.png', dpi=150)
plt.show()

print('\n── Metric Comparison ─────────────────────────────')
print(f'{"Metric":<20} {"Original":>10} {"Augmented":>10} {"Δ":>8}')
print('-' * 52)
for lbl, k in zip(labels_pretty, metric_keys):
    o, a = orig_metrics.get(k, 0), aug_metrics.get(k, 0)
    print(f'{lbl:<20} {o:>10.4f} {a:>10.4f} {a-o:>+8.4f}')

## 🔍 Step 8: Run Inference on Validation Images

In [ ]:
# ─── Inference visualisation ──────────────────────────────────────────────────
best_model = YOLO(str(BASE / 'best.pt'))
val_images  = list((YOLO_DIR / 'images' / 'val').glob('*.jpg'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, val_images):
    results = best_model.predict(str(img_path), conf=0.25, verbose=False)
    plot_img = results[0].plot()   # numpy array with boxes drawn
    ax.imshow(cv2.cvtColor(plot_img, cv2.COLOR_BGR2RGB))
    ax.set_title(img_path.name, fontsize=9)
    ax.axis('off')

plt.suptitle('YOLOv8 Logo Detection — Validation Samples',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE / 'inference_results.png', dpi=150)
plt.show()
print('✅ Inference visualisation saved')

## 📦 Step 9: Package Outputs

In [ ]:
# ─── Create downloadable zip ──────────────────────────────────────────────────
files_to_pack = [
    BASE / 'best.pt',
    BASE / 'generator.pth',
    BASE / 'gan_training_results.png',
    BASE / 'generated_samples.png',
    BASE / 'comparison_chart.png',
    BASE / 'inference_results.png',
]

zip_path = BASE / 'logo_detection_outputs.zip'
with zipfile.ZipFile(zip_path, 'w') as z:
    for f in files_to_pack:
        if f.exists():
            z.write(f, f.name)
            print(f'  + {f.name}')

print(f'\n✅ Download: {zip_path}')
print('\n📋 Summary:')
print(f'  - GAN trained for {GAN_EPOCHS} epochs')
print(f'  - {GENERATE_COUNT} synthetic images generated')
print(f'  - YOLOv8 trained (original + augmented)')
print(f'  - Best model: {BASE}/best.pt')